# Timbiriche (Dots and Boxes) — Minimax
**Inteligencia Artificial | TEC | Profesor: Kenneth Obando Rodríguez**

---

### Cómo leer el tablero
- Rayas azules = Jugador 1, rojas = Jugador 2
- Las rayas disponibles muestran su código (ej: `h01` = horizontal fila 0 col 1)
- Los cuadros completados muestran quién los cerró

### Cómo ingresar un movimiento
```
h 0 1   ->  raya horizontal, fila 0, columna 1
v 2 0   ->  raya vertical, fila 2, columna 0
ayuda   ->  ver todas las rayas disponibles
salir   ->  terminar
```

## Celda 1 — Todo el código
Ejecuta esta celda primero.

In [ ]:
import copy
import math
import time

# -------------------------------------------------------
# Representacion del tablero
#
# Uso tres listas de listas para guardar el estado:
#   rayas_h[f][c] -> quien puso la raya horizontal (0=nadie, 1=J1, 2=J2)
#   rayas_v[f][c] -> quien puso la raya vertical
#   cajas[f][c]   -> quien cerro la caja (0=abierta)
#
# Para un tablero de n cajas por lado:
#   rayas_h tiene n+1 filas y n columnas
#   rayas_v tiene n filas y n+1 columnas
# -------------------------------------------------------

class Tablero:

    def __init__(self, n=3):
        self.n = n
        self.rayas_h = [[0] * n for _ in range(n + 1)]
        self.rayas_v = [[0] * (n + 1) for _ in range(n)]
        self.cajas = [[0] * n for _ in range(n)]
        self.puntos = [0, 0]  # puntos[0] = J1, puntos[1] = J2
        self.turno = 1

    def copiar(self):
        return copy.deepcopy(self)

    def movimientos_posibles(self):
        # retorna todas las rayas que todavia no fueron puestas
        lista = []
        for f in range(self.n + 1):
            for c in range(self.n):
                if self.rayas_h[f][c] == 0:
                    lista.append(('h', f, c))
        for f in range(self.n):
            for c in range(self.n + 1):
                if self.rayas_v[f][c] == 0:
                    lista.append(('v', f, c))
        return lista

    def hacer_movimiento(self, mov):
        # aplica el movimiento en una copia y la retorna
        # asi el tablero original no cambia
        nuevo = self.copiar()
        tipo, f, c = mov

        if tipo == 'h':
            nuevo.rayas_h[f][c] = self.turno
        else:
            nuevo.rayas_v[f][c] = self.turno

        # revisar si se cerro alguna caja con esta raya
        cajas_cerradas = 0

        # una raya horizontal toca la caja de abajo (f,c) y la de arriba (f-1,c)
        # una raya vertical toca la caja a la derecha (f,c) y a la izquierda (f,c-1)
        if tipo == 'h':
            candidatos = [(f, c), (f - 1, c)]
        else:
            candidatos = [(f, c), (f, c - 1)]

        for bf, bc in candidatos:
            if 0 <= bf < self.n and 0 <= bc < self.n:
                if nuevo.cajas[bf][bc] == 0 and nuevo._esta_completa(bf, bc):
                    nuevo.cajas[bf][bc] = self.turno
                    cajas_cerradas += 1

        nuevo.puntos[self.turno - 1] += cajas_cerradas

        # si no cerro ninguna caja le toca al otro jugador
        # si cerro al menos una, el mismo jugador vuelve a jugar
        if cajas_cerradas == 0:
            nuevo.turno = 2 if self.turno == 1 else 1

        return nuevo, cajas_cerradas

    def _esta_completa(self, f, c):
        # una caja esta completa cuando tiene sus 4 lados
        top    = self.rayas_h[f][c]
        bottom = self.rayas_h[f + 1][c]
        left   = self.rayas_v[f][c]
        right  = self.rayas_v[f][c + 1]
        return top and bottom and left and right

    def contar_lados(self, f, c):
        top    = 1 if self.rayas_h[f][c] else 0
        bottom = 1 if self.rayas_h[f + 1][c] else 0
        left   = 1 if self.rayas_v[f][c] else 0
        right  = 1 if self.rayas_v[f][c + 1] else 0
        return top + bottom + left + right

    def juego_terminado(self):
        # el juego termina cuando todas las cajas estan cerradas
        for f in range(self.n):
            for c in range(self.n):
                if self.cajas[f][c] == 0:
                    return False
        return True

    def quien_gano(self):
        if self.puntos[0] > self.puntos[1]:
            return 1
        elif self.puntos[1] > self.puntos[0]:
            return 2
        else:
            return 0


# -------------------------------------------------------
# Visualizacion del tablero en ASCII
# -------------------------------------------------------

# codigos de color ANSI
RESET  = '\033[0m'
AZUL   = '\033[94m'
ROJO   = '\033[91m'
VERDE  = '\033[92m'
GRIS   = '\033[90m'
AMARILLO = '\033[93m'

def mostrar_tablero(t, ultimo=None):
    n = t.n
    filas = []

    for f in range(n + 1):
        # fila con rayas horizontales
        linea = '  '
        for c in range(n):
            quien = t.rayas_h[f][c]
            if quien == 0:
                linea += 'o' + GRIS + '-h' + str(f) + str(c) + '-' + RESET
            elif quien == 1:
                if ultimo == ('h', f, c):
                    linea += 'o' + VERDE + '-----' + RESET
                else:
                    linea += 'o' + AZUL + '-----' + RESET
            else:
                if ultimo == ('v', f, c):
                    linea += 'o' + VERDE + '=====' + RESET
                else:
                    linea += 'o' + ROJO + '=====' + RESET
        linea += 'o'
        filas.append(linea)

        # fila con rayas verticales y cajas
        if f < n:
            linea = '  '
            for c in range(n + 1):
                quien = t.rayas_v[f][c]
                if quien == 0:
                    linea += GRIS + ':' + RESET
                elif quien == 1:
                    if ultimo == ('v', f, c):
                        linea += VERDE + '|' + RESET
                    else:
                        linea += AZUL + '|' + RESET
                else:
                    if ultimo == ('v', f, c):
                        linea += VERDE + '|' + RESET
                    else:
                        linea += ROJO + '|' + RESET

                # interior de la caja
                if c < n:
                    dueno = t.cajas[f][c]
                    if dueno == 1:
                        linea += '\033[44m\033[97m' + '  1  ' + RESET
                    elif dueno == 2:
                        linea += '\033[41m\033[97m' + '  2  ' + RESET
                    else:
                        linea += '     '
            filas.append(linea)

    # mostrar puntaje
    filas.append('')
    estado = '  ' + AZUL + 'J1: ' + str(t.puntos[0]) + ' pts' + RESET
    estado += '   ' + ROJO + 'J2: ' + str(t.puntos[1]) + ' pts' + RESET
    if t.juego_terminado():
        g = t.quien_gano()
        if g == 0:
            estado += '   ' + AMARILLO + 'EMPATE' + RESET
        else:
            estado += '   ' + AMARILLO + 'Gano J' + str(g) + RESET
    else:
        if t.turno == 1:
            estado += '   Turno: ' + AZUL + 'J1' + RESET
        else:
            estado += '   Turno: ' + ROJO + 'J2' + RESET
    filas.append(estado)

    print('\n'.join(filas))


def mostrar_leyenda():
    print()
    print('  Leyenda:')
    print('    ' + AZUL + '-----' + RESET + '  raya J1')
    print('    ' + ROJO + '=====' + RESET + '  raya J2')
    print('    ' + GRIS + '-h01-' + RESET + '  raya disponible (codigo: h 0 1)')
    print('    ' + GRIS + ':' + RESET + '      raya vertical disponible')
    print()
    print('  Formato: h fila col  o  v fila col')
    print('  Ejemplo: h 0 1')
    print()


def mostrar_opciones(t):
    movs = t.movimientos_posibles()
    horiz = []
    verti = []
    for tipo, f, c in movs:
        if tipo == 'h':
            horiz.append('h ' + str(f) + ' ' + str(c))
        else:
            verti.append('v ' + str(f) + ' ' + str(c))
    print()
    print('  Rayas disponibles:')
    print('  Horizontales: ' + ', '.join(horiz))
    print('  Verticales:   ' + ', '.join(verti))
    print()


# -------------------------------------------------------
# Heuristica
#
# Estima que tan buena es la posicion para un jugador.
# Tome en cuenta:
#   - diferencia de puntos (lo mas importante)
#   - cajas que puedo cerrar ahora mismo (3 lados)
#   - cajas que puede cerrar el rival ahora
#   - cajas con 2 lados (peligrosas, conviene evitarlas)
# -------------------------------------------------------

def evaluar(t, jugador):
    rival = 2 if jugador == 1 else 1

    mis_puntos    = t.puntos[jugador - 1]
    puntos_rival  = t.puntos[rival - 1]
    diferencia    = (mis_puntos - puntos_rival) * 3

    cierro_yo  = 0
    cierra_el  = 0
    peligrosas = 0

    for f in range(t.n):
        for c in range(t.n):
            if t.cajas[f][c] != 0:
                continue
            lados = t.contar_lados(f, c)
            if lados == 3:
                if t.turno == jugador:
                    cierro_yo += 1
                else:
                    cierra_el += 1
            elif lados == 2:
                peligrosas += 1

    valor = diferencia + cierro_yo * 2 - cierra_el * 2 - peligrosas * 0.5
    return valor


# -------------------------------------------------------
# Minimax con poda alpha-beta
#
# Nota: en timbiriche el turno no siempre alterna.
# Si cerras una caja, volvés a jugar. Por eso uso
# t.turno para saber si es MAX o MIN en cada nodo,
# no la profundidad como en el minimax clasico.
# -------------------------------------------------------

nodos = 0  # para contar cuantos nodos se visitan

def minimax(t, profundidad, alpha, beta, jugador_max):
    global nodos
    nodos += 1

    # si el juego termino, retornar el resultado real
    if t.juego_terminado():
        diff = t.puntos[jugador_max - 1] - t.puntos[2 - jugador_max]
        return diff * 100

    # si llegamos al limite de busqueda, usar la heuristica
    if profundidad == 0:
        return evaluar(t, jugador_max)

    movs = t.movimientos_posibles()

    if t.turno == jugador_max:
        # turno de MAX: quiere el valor mas alto
        mejor = -math.inf
        for mov in movs:
            hijo, _ = t.hacer_movimiento(mov)
            val = minimax(hijo, profundidad - 1, alpha, beta, jugador_max)
            if val > mejor:
                mejor = val
            if mejor > alpha:
                alpha = mejor
            if alpha >= beta:
                break  # poda
        return mejor
    else:
        # turno de MIN: quiere el valor mas bajo
        mejor = math.inf
        for mov in movs:
            hijo, _ = t.hacer_movimiento(mov)
            val = minimax(hijo, profundidad - 1, alpha, beta, jugador_max)
            if val < mejor:
                mejor = val
            if mejor < beta:
                beta = mejor
            if alpha >= beta:
                break  # poda
        return mejor


def elegir_movimiento(t, profundidad=5):
    global nodos
    nodos = 0

    jugador = t.turno
    mejor_mov = None
    mejor_val = -math.inf

    inicio = time.time()
    for mov in t.movimientos_posibles():
        hijo, _ = t.hacer_movimiento(mov)
        val = minimax(hijo, profundidad - 1, -math.inf, math.inf, jugador)
        if val > mejor_val:
            mejor_val = val
            mejor_mov = mov
    fin = time.time()

    return mejor_mov, mejor_val, fin - inicio, nodos


# -------------------------------------------------------
# Leer el movimiento que escribe el usuario
# -------------------------------------------------------

def leer_jugada(texto, t):
    texto = texto.strip().lower()
    texto = texto.replace(',', ' ')
    partes = texto.split()

    if len(partes) == 3:
        tipo = partes[0]
        try:
            f = int(partes[1])
            c = int(partes[2])
        except:
            return None
    elif len(partes) == 1 and len(texto) >= 3:
        tipo = texto[0]
        try:
            f = int(texto[1])
            c = int(texto[2])
        except:
            return None
    else:
        return None

    if tipo != 'h' and tipo != 'v':
        return None

    mov = (tipo, f, c)
    if mov in t.movimientos_posibles():
        return mov
    return None


# -------------------------------------------------------
# Modo 1: Humano vs Maquina
# -------------------------------------------------------

def humano_vs_maquina(n=3, profundidad_ia=5, ia_juega_como=2):
    humano = 1 if ia_juega_como == 2 else 2
    t = Tablero(n)

    registro = []
    tiempos_ia = []
    nodos_ia = []

    print()
    print('=' * 50)
    print('  TIMBIRICHE - Humano vs Maquina')
    print('  Tablero ' + str(n) + 'x' + str(n))
    print('  Eres el Jugador ' + str(humano))
    print('  IA es el Jugador ' + str(ia_juega_como))
    print('=' * 50)
    mostrar_leyenda()

    ultimo = None

    while not t.juego_terminado():
        print()
        mostrar_tablero(t, ultimo)
        print()

        if t.turno == ia_juega_como:
            print('  IA pensando...')
            mov, val, tiempo, n_nodos = elegir_movimiento(t, profundidad_ia)
            tiempos_ia.append(tiempo)
            nodos_ia.append(n_nodos)

            tipo, f, c = mov
            print('  IA juega: ' + tipo + ' ' + str(f) + ' ' + str(c))
            print('  (' + str(round(tiempo, 2)) + 's, ' + str(n_nodos) + ' nodos)')

            t, cerradas = t.hacer_movimiento(mov)
            ultimo = mov
            registro.append((ia_juega_como, mov, cerradas))
            if cerradas > 0:
                print('  La IA cerro ' + str(cerradas) + ' caja(s). Vuelve a jugar.')

        else:
            mostrar_opciones(t)
            while True:
                try:
                    entrada = input('  Tu jugada (J' + str(humano) + '): ')
                except EOFError:
                    entrada = 'salir'

                if entrada.lower() == 'salir':
                    print('  Partida terminada.')
                    return t, registro, tiempos_ia, nodos_ia

                if entrada.lower() == 'ayuda':
                    mostrar_opciones(t)
                    continue

                mov = leer_jugada(entrada, t)
                if mov is None:
                    print('  Movimiento invalido. Escribe ayuda para ver las opciones.')
                    continue

                t, cerradas = t.hacer_movimiento(mov)
                ultimo = mov
                registro.append((humano, mov, cerradas))
                if cerradas > 0:
                    print('  Cerraste ' + str(cerradas) + ' caja(s). Vuelves a jugar.')
                break

    print()
    mostrar_tablero(t, ultimo)
    mostrar_resultado(t, humano, ia_juega_como)
    return t, registro, tiempos_ia, nodos_ia


# -------------------------------------------------------
# Modo 2: Maquina vs Maquina
# -------------------------------------------------------

def maquina_vs_maquina(n=3, prof_j1=5, prof_j2=5, pausa=0.5):
    from IPython.display import clear_output

    t = Tablero(n)
    registro = []
    tiempos = {1: [], 2: []}
    nodos_por_turno = {1: [], 2: []}
    profundidades = {1: prof_j1, 2: prof_j2}

    print()
    print('=' * 50)
    print('  TIMBIRICHE - Maquina vs Maquina')
    print('  Tablero ' + str(n) + 'x' + str(n))
    print('  J1 profundidad=' + str(prof_j1) + '  J2 profundidad=' + str(prof_j2))
    print('=' * 50)

    num_turno = 0
    ultimo = None

    while not t.juego_terminado():
        num_turno += 1
        j = t.turno
        prof = profundidades[j]

        clear_output(wait=True)
        print('\n  Turno ' + str(num_turno) + ' - J' + str(j) + ' pensando...\n')
        mostrar_tablero(t, ultimo)

        mov, val, tiempo, n_nodos = elegir_movimiento(t, prof)
        tiempos[j].append(tiempo)
        nodos_por_turno[j].append(n_nodos)

        tipo, f, c = mov
        print('\n  J' + str(j) + ' juega: ' + tipo + ' ' + str(f) + ' ' + str(c))
        print('  valor=' + str(round(val, 1)) + '  tiempo=' + str(round(tiempo, 2)) + 's  nodos=' + str(n_nodos))

        t, cerradas = t.hacer_movimiento(mov)
        ultimo = mov
        registro.append((j, mov, cerradas))

        if cerradas > 0:
            print('  J' + str(j) + ' cerro ' + str(cerradas) + ' caja(s). Vuelve a jugar.')

        if pausa > 0:
            time.sleep(pausa)

    clear_output(wait=True)
    print()
    mostrar_tablero(t, ultimo)
    mostrar_resultado(t)

    print()
    print('  Estadisticas:')
    for j in [1, 2]:
        if len(tiempos[j]) > 0:
            total = sum(tiempos[j])
            prom  = total / len(tiempos[j])
            tot_nodos = sum(nodos_por_turno[j])
            print('  J' + str(j) + ': ' + str(round(total, 2)) + 's total, '
                  + str(round(prom, 3)) + 's promedio, '
                  + str(tot_nodos) + ' nodos')

    return t, registro, tiempos, nodos_por_turno


def mostrar_resultado(t, humano=None, ia=None):
    print()
    print('=' * 50)
    print('  RESULTADO FINAL')
    print('  J1: ' + str(t.puntos[0]) + ' cajas')
    print('  J2: ' + str(t.puntos[1]) + ' cajas')
    g = t.quien_gano()
    if g == 0:
        print('  ' + AMARILLO + 'EMPATE' + RESET)
    elif humano is not None and g == humano:
        print('  ' + AMARILLO + 'Ganaste! (J' + str(g) + ')' + RESET)
    elif ia is not None and g == ia:
        print('  ' + AMARILLO + 'Gano la IA (J' + str(g) + ')' + RESET)
    else:
        print('  ' + AMARILLO + 'Gano J' + str(g) + RESET)
    print('=' * 50)


# -------------------------------------------------------
# Pruebas para verificar que todo funciona
# -------------------------------------------------------

def correr_pruebas():
    errores = 0

    print('\n' + '-' * 45)
    print('  Pruebas')
    print('-' * 45)

    # prueba 1: tablero 2x2 vacio tiene 12 rayas disponibles
    # 3 filas x 2 cols horizontales = 6, mas 2 filas x 3 cols verticales = 6
    t = Tablero(2)
    if len(t.movimientos_posibles()) == 12:
        print('  OK - tablero 2x2: 12 movimientos iniciales')
    else:
        print('  FALLO - tablero 2x2: ' + str(len(t.movimientos_posibles())) + ' movimientos (esperaba 12)')
        errores += 1

    # prueba 2: despues de poner una raya quedan 11
    t2, _ = t.hacer_movimiento(('h', 0, 0))
    if len(t2.movimientos_posibles()) == 11:
        print('  OK - despues de un movimiento quedan 11')
    else:
        print('  FALLO - deberian quedar 11')
        errores += 1

    # prueba 3: el tablero original no debe cambiar
    if t.rayas_h[0][0] == 0:
        print('  OK - el original no cambia al copiar')
    else:
        print('  FALLO - el original cambio')
        errores += 1

    # prueba 4: cerrar una caja
    # J1 pone h0,0 -> turno pasa a J2
    # J2 pone h1,0 -> turno pasa a J1
    # J1 pone v0,0 -> turno pasa a J2
    # J2 pone v0,1 -> cierra la caja, J2 vuelve a jugar
    t = Tablero(2)
    t, _ = t.hacer_movimiento(('h', 0, 0))
    t, _ = t.hacer_movimiento(('h', 1, 0))
    t, _ = t.hacer_movimiento(('v', 0, 0))
    t, cerradas = t.hacer_movimiento(('v', 0, 1))
    if cerradas == 1 and t.puntos[1] == 1 and t.cajas[0][0] == 2:
        print('  OK - cerrar caja: punto a J2')
    else:
        print('  FALLO - cerrar caja no funciona bien')
        errores += 1

    # prueba 5: el turno no cambia cuando se cierra una caja
    if t.turno == 2:
        print('  OK - turno no cambia al cerrar caja')
    else:
        print('  FALLO - el turno cambio cuando no debia')
        errores += 1

    # prueba 6: detectar fin del juego
    t = Tablero(2)
    while not t.juego_terminado():
        movs = t.movimientos_posibles()
        t, _ = t.hacer_movimiento(movs[0])
    if t.juego_terminado() and sum(t.puntos) == 4:
        print('  OK - fin del juego detectado, 4 puntos en total')
    else:
        print('  FALLO - no detecto el fin del juego')
        errores += 1

    # prueba 7: minimax tiene que elegir cerrar la caja obvia
    # ponemos 3 lados de la caja (0,0), el cuarto lado es v(0,1)
    t = Tablero(2)
    t.rayas_h[0][0] = 1
    t.rayas_h[1][0] = 1
    t.rayas_v[0][0] = 1
    t.turno = 1
    mov, _, _, _ = elegir_movimiento(t, 3)
    if mov == ('v', 0, 1):
        print('  OK - minimax elige cerrar la caja (v 0 1)')
    else:
        print('  FALLO - minimax eligio ' + str(mov) + ' en vez de v 0 1')
        errores += 1

    print('-' * 45)
    if errores == 0:
        print('  Todas las pruebas pasaron')
    else:
        print('  ' + str(errores) + ' pruebas fallaron')
    print('-' * 45)


# -------------------------------------------------------
# Mostrar el camino de la partida
# -------------------------------------------------------

def mostrar_camino(registro, n, cada_cuantos=5):
    print()
    print('=' * 50)
    print('  Camino de la partida')
    print('=' * 50)

    t = Tablero(n)
    print('\n  Estado inicial:')
    mostrar_tablero(t)

    for i in range(len(registro)):
        jugador, mov, cerradas = registro[i]
        tipo, f, c = mov
        t, _ = t.hacer_movimiento(mov)

        if tipo == 'h':
            dir_str = 'horizontal'
        else:
            dir_str = 'vertical'

        print('\n  Mov ' + str(i + 1) + ': J' + str(jugador) + ' pone raya ' + dir_str + ' (' + tipo + ' ' + str(f) + ' ' + str(c) + ')', end='')
        if cerradas > 0:
            print(' -> cerro ' + str(cerradas) + ' caja(s)', end='')
        print()
        print('  J1=' + str(t.puntos[0]) + '  J2=' + str(t.puntos[1]))

        # mostrar tablero cada ciertos movimientos o cuando se cierra una caja
        if cada_cuantos > 0 and (i + 1) % cada_cuantos == 0:
            mostrar_tablero(t, mov)
        elif cerradas > 0:
            mostrar_tablero(t, mov)

    print()
    print('  Estado final:')
    mostrar_tablero(t)
    g = t.quien_gano()
    print('\n  Total de movimientos: ' + str(len(registro)))
    if g == 0:
        print('  Resultado: empate')
    else:
        print('  Resultado: gano J' + str(g))
    print('=' * 50)


# -------------------------------------------------------
# Tiempo de ejecucion
# -------------------------------------------------------

def mostrar_tiempos(registro, tiempos=None, nodos_lista=None, tiempos_mvm=None, nodos_mvm=None):
    print()
    print('-' * 45)
    print('  Tiempo de ejecucion del algoritmo')
    print('-' * 45)
    print('  Total de movimientos: ' + str(len(registro)))

    # para maquina vs maquina
    if tiempos_mvm is not None:
        for j in [1, 2]:
            ts = tiempos_mvm[j]
            ns = nodos_mvm[j]
            if len(ts) > 0:
                print()
                print('  J' + str(j) + ':')
                print('    Turnos:   ' + str(len(ts)))
                print('    Total:    ' + str(round(sum(ts), 3)) + ' s')
                print('    Promedio: ' + str(round(sum(ts) / len(ts), 4)) + ' s')
                print('    Maximo:   ' + str(round(max(ts), 4)) + ' s')
                print('    Nodos:    ' + str(sum(ns)) + ' total')

    # para humano vs maquina
    if tiempos is not None and len(tiempos) > 0:
        print()
        print('  IA:')
        print('    Turnos:   ' + str(len(tiempos)))
        print('    Total:    ' + str(round(sum(tiempos), 3)) + ' s')
        print('    Promedio: ' + str(round(sum(tiempos) / len(tiempos), 4)) + ' s')
        print('    Maximo:   ' + str(round(max(tiempos), 4)) + ' s')
        if nodos_lista is not None:
            print('    Nodos:    ' + str(sum(nodos_lista)) + ' total')

    print('-' * 45)


# -------------------------------------------------------
# Graficas con matplotlib
# -------------------------------------------------------

def graficar_partida(registro, tiempos_mvm=None, nodos_mvm=None, tiempos_ia=None, nodos_ia=None, titulo='Timbiriche'):
    import matplotlib.pyplot as plt

    # evolucion del marcador
    p1 = [0]
    p2 = [0]
    for jugador, mov, cerradas in registro:
        if jugador == 1:
            p1.append(p1[-1] + cerradas)
            p2.append(p2[-1])
        else:
            p1.append(p1[-1])
            p2.append(p2[-1] + cerradas)

    if tiempos_mvm is not None:
        # graficas para maquina vs maquina
        fig, axes = plt.subplots(1, 3, figsize=(15, 4))
        fig.suptitle(titulo)

        # tiempo por turno
        ts1 = tiempos_mvm[1]
        ts2 = tiempos_mvm[2]
        xs1 = [i for i, (j, _, _) in enumerate(registro) if j == 1]
        xs2 = [i for i, (j, _, _) in enumerate(registro) if j == 2]
        axes[0].bar(xs1, ts1, color='#4A90D9', label='J1')
        axes[0].bar(xs2, ts2, color='#E05A5A', label='J2')
        axes[0].set_title('Tiempo por turno')
        axes[0].set_xlabel('Turno')
        axes[0].set_ylabel('Segundos')
        axes[0].legend()
        axes[0].grid(axis='y', alpha=0.3)

        # nodos por turno
        ns1 = nodos_mvm[1]
        ns2 = nodos_mvm[2]
        axes[1].plot(xs1, ns1, 'o-', color='#4A90D9', label='J1')
        axes[1].plot(xs2, ns2, 's-', color='#E05A5A', label='J2')
        axes[1].set_title('Nodos explorados por turno')
        axes[1].set_xlabel('Turno')
        axes[1].set_ylabel('Nodos')
        axes[1].set_yscale('log')
        axes[1].legend()
        axes[1].grid(alpha=0.3)

        # marcador
        axes[2].step(range(len(p1)), p1, where='post', color='#4A90D9', label='J1', linewidth=2)
        axes[2].step(range(len(p2)), p2, where='post', color='#E05A5A', label='J2', linewidth=2)
        axes[2].set_title('Evolucion del marcador')
        axes[2].set_xlabel('Movimiento')
        axes[2].set_ylabel('Cajas')
        axes[2].legend()
        axes[2].grid(alpha=0.3)

        plt.tight_layout()
        plt.show()

    elif tiempos_ia is not None:
        # graficas para humano vs maquina
        fig, axes = plt.subplots(1, 3, figsize=(15, 4))
        fig.suptitle(titulo)

        # tiempo por turno de la ia
        axes[0].bar(range(len(tiempos_ia)), tiempos_ia, color='#E05A5A')
        if len(tiempos_ia) > 0:
            prom = sum(tiempos_ia) / len(tiempos_ia)
            axes[0].axhline(prom, color='black', linestyle='--', label='promedio ' + str(round(prom, 3)) + 's')
            axes[0].legend()
        axes[0].set_title('Tiempo por turno (IA)')
        axes[0].set_xlabel('Turno de la IA')
        axes[0].set_ylabel('Segundos')
        axes[0].grid(axis='y', alpha=0.3)

        # nodos
        if nodos_ia is not None:
            axes[1].plot(range(len(nodos_ia)), nodos_ia, 'o-', color='#9B59B6')
            axes[1].set_title('Nodos explorados (IA)')
            axes[1].set_xlabel('Turno')
            axes[1].set_ylabel('Nodos')
            axes[1].set_yscale('log')
            axes[1].grid(alpha=0.3)

        # marcador
        axes[2].step(range(len(p1)), p1, where='post', color='#4A90D9', label='J1', linewidth=2)
        axes[2].step(range(len(p2)), p2, where='post', color='#E05A5A', label='J2', linewidth=2)
        axes[2].set_title('Evolucion del marcador')
        axes[2].set_xlabel('Movimiento')
        axes[2].set_ylabel('Cajas')
        axes[2].legend()
        axes[2].grid(alpha=0.3)

        plt.tight_layout()
        plt.show()


print('Listo. Ejecuta la Celda 2 para probar que todo funciona.')


## Celda 2 — Pruebas
Verifica que la implementación esta correcta.

In [ ]:
correr_pruebas()

## Celda 3 — Maquina vs Maquina
Dos agentes Minimax juegan entre si. `N` es la cantidad de cajas por lado (N=3 crea un tablero de 3x3 cajas).

In [ ]:
N      = 3    # cajas por lado
PROF1  = 5    # profundidad de busqueda J1
PROF2  = 5    # profundidad de busqueda J2
PAUSA  = 0.5  # segundos entre movimientos

t_final, reg, tiempos_mvm, nodos_mvm = maquina_vs_maquina(
    n=N, prof_j1=PROF1, prof_j2=PROF2, pausa=PAUSA
)

### Tiempo de ejecucion

In [ ]:
mostrar_tiempos(reg, tiempos_mvm=tiempos_mvm, nodos_mvm=nodos_mvm)

### Camino de la partida
Muestra el tablero cada 5 movimientos y siempre que se cierra una caja.

In [ ]:
mostrar_camino(reg, n=N, cada_cuantos=5)

### Graficas

In [ ]:
graficar_partida(reg, tiempos_mvm=tiempos_mvm, nodos_mvm=nodos_mvm, titulo='Maquina vs Maquina')

## Celda 4 — Humano vs Maquina
Juega contra la IA. Mira las etiquetas en el tablero y escribe el codigo de la raya.

In [ ]:
N          = 3  # cajas por lado
PROF_IA    = 5  # profundidad de la IA
IA_JUGADOR = 2  # la IA juega como J2

t_hvm, reg_hvm, tiempos_ia, nodos_ia = humano_vs_maquina(
    n=N, profundidad_ia=PROF_IA, ia_juega_como=IA_JUGADOR
)

### Tiempo de ejecucion

In [ ]:
if reg_hvm:
    mostrar_tiempos(reg_hvm, tiempos=tiempos_ia, nodos_lista=nodos_ia)

### Camino de la partida

In [ ]:
if reg_hvm:
    mostrar_camino(reg_hvm, n=N, cada_cuantos=1)

### Graficas

In [ ]:
if reg_hvm and len(tiempos_ia) > 0:
    graficar_partida(reg_hvm, tiempos_ia=tiempos_ia, nodos_ia=nodos_ia,
                     titulo='Humano vs Maquina')

## Celda 5 — Benchmark
Mide como crece el tiempo con la profundidad.

In [ ]:
import matplotlib.pyplot as plt

def benchmark(n_tablero=2, profundidades=None):
    if profundidades is None:
        profundidades = [1, 2, 3, 4, 5, 6, 7]

    t = Tablero(n_tablero)
    lista_tiempos = []
    lista_nodos = []

    print('Benchmark tablero ' + str(n_tablero) + 'x' + str(n_tablero))
    print('  prof   tiempo        nodos')
    print('  ' + '-'*32)

    for p in profundidades:
        _, _, dur, n = elegir_movimiento(t, p)
        lista_tiempos.append(dur)
        lista_nodos.append(n)
        print('  ' + str(p) + '      ' + str(round(dur, 4)) + '        ' + str(n))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
    fig.suptitle('Escalabilidad del Minimax - tablero ' + str(n_tablero) + 'x' + str(n_tablero))

    ax1.plot(profundidades, lista_tiempos, 'o-', color='#4A90D9')
    ax1.set_title('Tiempo de busqueda')
    ax1.set_xlabel('Profundidad')
    ax1.set_ylabel('Segundos')
    ax1.grid(alpha=0.3)

    ax2.plot(profundidades, lista_nodos, 's-', color='#E05A5A')
    ax2.set_title('Nodos explorados')
    ax2.set_xlabel('Profundidad')
    ax2.set_ylabel('Nodos')
    ax2.set_yscale('log')
    ax2.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

benchmark(n_tablero=2, profundidades=[1, 2, 3, 4, 5, 6, 7])

## Celda 6 — Experimento: IA fuerte vs IA debil

In [ ]:
from IPython.display import clear_output

print('J1 (profundidad=2) vs J2 (profundidad=5)')
t_exp, reg_exp, t_exp_mvm, n_exp_mvm = maquina_vs_maquina(n=3, prof_j1=2, prof_j2=5, pausa=0.2)
graficar_partida(reg_exp, tiempos_mvm=t_exp_mvm, nodos_mvm=n_exp_mvm, titulo='IA debil vs IA fuerte')

---
## Informe

### Representacion del estado
El tablero se guarda con tres listas de listas: `rayas_h`, `rayas_v` y `cajas`. Cada posicion guarda quien puso esa raya (0=nadie, 1=J1, 2=J2). Para hacer un movimiento se copia todo el tablero con `copy.deepcopy` y se modifica la copia, asi el estado anterior queda intacto para que el algoritmo pueda explorar otras ramas.

Una caja queda cerrada cuando los 4 lados (`rayas_h[f][c]`, `rayas_h[f+1][c]`, `rayas_v[f][c]`, `rayas_v[f][c+1]`) estan puestos.

### Regla del turno
En Timbiriche quien cierra una caja vuelve a jugar. Esto complica el Minimax porque el turno no siempre alterna. La solucion fue revisar `t.turno` en cada nodo para saber si es turno de MAX o MIN, en lugar de usar la paridad de la profundidad como se hace en el Minimax clasico.

### Heuristica
La funcion `evaluar()` combina cuatro cosas:

| Factor | Peso |
|:-------|:----:|
| Diferencia de puntos | x3 |
| Cajas que puedo cerrar ahora (3 lados) | +2 |
| Cajas que puede cerrar el rival ahora | -2 |
| Cajas con 2 lados (peligrosas) | -0.5 |

La logica es que nunca conviene poner el tercer lado de una caja si no la vas a cerrar vos, porque el rival la cierra en su proximo movimiento. La penalizacion por cajas con 2 lados captura eso.

### Minimax con poda Alpha-Beta
- `alpha` es el mejor valor que MAX puede garantizar
- `beta` es el mejor valor que MIN puede garantizar
- Si `alpha >= beta` se corta la busqueda en esa rama porque no puede mejorar el resultado
- Esto reduce la cantidad de nodos que hay que explorar sin cambiar el resultado